# ToolUse Agent using Anthropic models

### State Diagram (Agent View)
```mermaid
stateDiagram-v2
direction TB

    INIT --> HAS_MESSAGE
    HAS_MESSAGE --> CHAT: condition_true
    HAS_MESSAGE --> TOOL_USE: condition_false

    CHAT--> IS_TOOL_CALL
    
    IS_TOOL_CALL --> FINAL: condition_false
    IS_TOOL_CALL --> TOOL_USE: condition_true

    TOOL_USE --> FINAL


```

### State Diagram (User View)
```mermaid
stateDiagram-v2
direction TB
    state "Start Agent" as StartAgent
    state "Continue Agent" as ContinueAgent
    state "Interrupt Agent" as InterruptAgent

    Initialize --> StartAgent : start_async
    StartAgent --> ContinueAgent : continue_async
    ContinueAgent --> InterruptAgent : interrupt_async
    InterruptAgent --> ContinueAgent : continue_async
    ContinueAgent --> ContinueAgent : continue_async
    ContinueAgent --> Terminate
    Terminate --> StartAgent : start_async
    Terminate --> InterruptAgent : interrupt_async

```



### a) Create Agent

In [1]:
from gai.asm.agents import ToolUseAgent
from gai.mcp.client.mcp_client import McpAggregatedClient
from gai.lib.config import config_helper

from gai.lib.tests import make_local_tmp
import os
here = make_local_tmp()
file_path = os.path.join(here, "monologue.json")
from gai.messages import FileMonologue
monologue = FileMonologue(agent_name="ToolUseAgent",file_path=file_path)

aggregated_client = McpAggregatedClient(["mcp-pseudo","mcp-time", "mcp-web"])
tools = await aggregated_client.list_tools()

# Create an artificial dialogue history for testing
from gai.messages import FileDialogue, MessagePydantic
messages = [
    MessagePydantic(**{
        'id': 'b1e5f98c-f6eb-47de-a6e2-387510d970f9', 
        'header': {
            'sender': 'User',
            'recipient': 'Sara',
            "timestamp": 1751308157.270983,
            "order": 0            
        }, 'body': {
            'type': 'chat.send',
            'dialogue_id': '00000000-0000-0000-0000-000000000000',
            'round_no': 0,
            'step_no': 0,
            'role': "user",
            'content': 'It is a very nice weather in Singapore right now.',
        }
    }),
    MessagePydantic(**{
        'id': 'abbc7961-45dc-4973-aaf4-a6224ed35d37', 
        'header': {
            'sender': 'Sara',
            'recipient': 'User',
            "timestamp": 1751308167.3488164,
            "order": 1
        }, 'body': {
            'type': 'chat.reply',
            'dialogue_id': '00000000-0000-0000-0000-000000000000',
            'round_no': 0, 
            'step_no': 1,
            'chunk_no':10,
            'chunk':'<eom>',
            'role': "assistant",
            'content': 'Yes, it is! The weather in Singapore is typically warm and humid, with occasional rain showers. It\'s a great time to enjoy outdoor activities or relax indoors with a cool drink. How can I assist you today?'
        }
    })]
from gai.lib.constants import DEFAULT_GUID
file_path = os.path.join(here, f"{DEFAULT_GUID}.json")
dialogue = FileDialogue(messages=messages,file_path=file_path)
recap = dialogue.extract_recap()

agent = ToolUseAgent(
    agent_name="ToolUseAgent",
    llm_config=config_helper.get_client_config(
        {
            "client_type": "anthropic",
            "model": "claude-sonnet-4-20250514",
            "extra": {
                "max_tokens": 32000,
                "temperature": 0.7,
                "top_p": 0.95,
                "tools": True,
                "stream": True,
            },
        }
    ),
    aggregated_client=aggregated_client,
    monologue=monologue,
    recap=recap
)

### reset monologue (optional)

In [2]:
monologue.reset()


### b) start_async

The location of the public holiday is inferred from the dialogue context.

In [3]:
goal = "When is the next public holiday?"
user_message=f"""
## Goal:
{goal}
## Instructions:
- You have the following tools at your disposal: {tools}
"""

resp=await agent.start_async(user_message=user_message)
# Stream the response
async for chunk in resp():
    if chunk:
        if isinstance(chunk, str):
            print(chunk, end="", flush=True)

I'll help you find the next public holiday in Singapore. Let me search for the current Singapore public holidays to give you the most up-to-date information.

### c) Show monologue

In [4]:
import json

# Show the monologue
print("\n───────────────────────── MONOLOGUE START ─────────────────────────")
messages = agent.fsm.monologue.list_messages()
for message in messages[-2:]:
    print(json.dumps(message.model_dump(), indent=4))
print("───────────────────────── MONOLOGUE END ─────────────────────────\n")

# Print memory size
mem_size = agent.fsm.monologue.get_total_size()
print("Total char size=", mem_size)


───────────────────────── MONOLOGUE START ─────────────────────────
{
    "id": "305f7771-ddb1-4790-80b9-416e7e7eb98d",
    "header": {
        "sender": "User",
        "recipient": "ToolUseAgent",
        "timestamp": 1752484438.739943,
        "order": 2
    },
    "body": {
        "type": "monologue",
        "state_name": "TOOL_USE",
        "step_no": 4,
        "content_type": "text",
        "role": "user",
        "content": [
            {
                "type": "tool_result",
                "tool_use_id": "toolu_01M8cqYfGF8FmgHWUMhibHFn",
                "content": "Current UTC time is 2025-07-14, and the time in Asia/Singapore is 2025-07-14."
            }
        ]
    }
}
{
    "id": "81e23a23-3a5b-48e7-b3f6-acb4611079b2",
    "header": {
        "sender": "ToolUseAgent",
        "recipient": "User",
        "timestamp": 1752484442.5150645,
        "order": 3
    },
    "body": {
        "type": "monologue",
        "state_name": "TOOL_USE",
        "step_no": 4,
    

### d) continue_async

In [5]:
resp = await agent.continue_async()
# Stream the response
async for chunk in resp():
    if chunk:
        if isinstance(chunk, str):
            print(chunk, end="", flush=True)


Based on the current date (July 14, 2025) and the official public holiday information from Singapore's Ministry of Manpower, **the next public holiday in Singapore is:**

**National Day - Saturday, August 9, 2025**

This is Singapore's National Day, which commemorates the country's independence. Since it falls on a Saturday, if your rest day normally falls on Saturday, you would be entitled to a replacement public holiday on Monday, August 10, 2025.

After National Day, the subsequent public holidays in 2025 are:
- **Deepavali** - Monday, October 20, 2025
- **Christmas Day** - Thursday, December 25, 2025

So you have about 3-4 weeks until the next public holiday! Perfect timing to plan something special for National Day celebrations.

### e) interrupt_async

Should be able to interrupt the agent and continue with original task.

In [7]:
resp = await agent.interrupt_async(user_message="Tell me a one paragraph joke.")
# Stream the response
async for chunk in resp():
    if chunk:
        if isinstance(chunk, str):
            print(chunk, end="", flush=True)


Here's a quick joke for you:

A man walks into a library and asks for books on paranoia. The librarian whispers, "They're right behind you!" The man spins around frantically, but there's nothing there. The librarian chuckles and points to the shelf directly behind him, saying, "No, seriously, the paranoia section is right behind you on that shelf." The man, still catching his breath, realizes he's been had and starts laughing. He grabs a book titled "How to Overcome Paranoia" and heads to the checkout counter, but then stops and asks, "Wait, how do I know you're not all in on this together?" The librarian just smiles and stamps his book with extra enthusiasm.

Now, getting back to our earlier conversation about Singapore's lovely weather and the upcoming public holidays - since National Day is coming up on August 9th, are you planning anything special for the long weekend? It's always a great time to enjoy Singapore's outdoor attractions or catch the National Day Parade if you're into 

### f) Show dialogue

In [8]:
dialogue.add_user_message(recipient='Sara', content=goal)
dialogue.add_assistant_message(sender='Sara', chunk="<eom>", content=agent.final_output())
for msg in dialogue.list_messages():
    print(f"{msg.header.sender}: {msg.body.content}")

User: It is a very nice weather in Singapore right now.
Sara: Yes, it is! The weather in Singapore is typically warm and humid, with occasional rain showers. It's a great time to enjoy outdoor activities or relax indoors with a cool drink. How can I assist you today?
User: Sara, When is the next public holiday?
Sara: Here's a quick joke for you:

A man walks into a library and asks for books on paranoia. The librarian whispers, "They're right behind you!" The man spins around frantically, but there's nothing there. The librarian chuckles and points to the shelf directly behind him, saying, "No, seriously, the paranoia section is right behind you on that shelf." The man, still catching his breath, realizes he's been had and starts laughing. He grabs a book titled "How to Overcome Paranoia" and heads to the checkout counter, but then stops and asks, "Wait, how do I know you're not all in on this together?" The librarian just smiles and stamps his book with extra enthusiasm.

Now, getting

---
## Scenario 1

- LLM Interrupts itself to ask user question
- User cannot continue because LLM is waiting for user input

In [1]:
import os
from gai.asm.agents import ToolUseAgent
from gai.mcp.client.mcp_client import McpAggregatedClient
from gai.lib.config import config_helper
from gai.lib.tests import make_local_tmp
from gai.messages import FileMonologue

here = make_local_tmp()
file_path = os.path.join(here, "monologue.json")
monologue = FileMonologue(agent_name="ToolUseAgent",file_path=file_path)
aggregated_client = McpAggregatedClient(["mcp-pseudo","mcp-time", "mcp-web"])
tools = await aggregated_client.list_tools()
agent = ToolUseAgent(
    agent_name="ToolUseAgent",
    project_name="gai",
    llm_config=config_helper.get_client_config(
        {
            "client_type": "anthropic",
            "model": "claude-sonnet-4-20250514",
            "extra": {
                "max_tokens": 32000,
                "temperature": 0.7,
                "top_p": 0.95,
                "tools": True,
                "stream": True,
            },
        }
    ),
    monologue=monologue,
    aggregated_client=aggregated_client,
)

In [2]:
monologue.reset()

goal = "What time is it right now?"
user_message = f"""
## Goal:
{goal}
## Instructions:
- You have the following tools at your disposal: {tools}
- Use "user_input" to ask for timezone before you start.
"""

resp = await agent.start_async(user_message=user_message)
# Stream the response
async for chunk in resp():
    if chunk:
        if isinstance(chunk, str):
            print(chunk, end="", flush=True)


I'll help you get the current time! First, let me ask you about your timezone preference.What timezone would you like me to show the current time for? Please provide the timezone in IANA format (e.g., America/New_York, Europe/London, Asia/Tokyo, UTC) or let me know your location and I can determine the appropriate timezone.

Confirm that "continue_async" will not work.

In [3]:
resp = await agent.continue_async()
# Stream the response
async for chunk in resp():
    if chunk:
        if isinstance(chunk, str):
            print(chunk, end="", flush=True)


---
## Scenario 2

- LLM Interrupts itself to ask user question
- User responds with answer

In [1]:
import os
from gai.asm.agents import ToolUseAgent
from gai.mcp.client.mcp_client import McpAggregatedClient
from gai.lib.config import config_helper
from gai.lib.tests import make_local_tmp
from gai.messages import FileMonologue

here = make_local_tmp()
file_path = os.path.join(here, "monologue.json")
monologue = FileMonologue(agent_name="ToolUseAgent",file_path=file_path)
aggregated_client = McpAggregatedClient(["mcp-pseudo","mcp-time", "mcp-web"])
tools = await aggregated_client.list_tools()
agent = ToolUseAgent(
    agent_name="ToolUseAgent",
    project_name="gai",
    llm_config=config_helper.get_client_config(
        {
            "client_type": "anthropic",
            "model": "claude-sonnet-4-20250514",
            "extra": {
                "max_tokens": 32000,
                "temperature": 0.7,
                "top_p": 0.95,
                "tools": True,
                "stream": True,
            },
        }
    ),
    monologue=monologue,
    aggregated_client=aggregated_client,
)

In [2]:
monologue.reset()

goal = "What time is it right now?"
user_message = f"""
## Goal:
{goal}
## Instructions:
- You have the following tools at your disposal: {tools}
- Use "user_input" to ask for timezone before you start.
"""

resp = await agent.start_async(user_message=user_message)
# Stream the response
async for chunk in resp():
    if chunk:
        if isinstance(chunk, str):
            print(chunk, end="", flush=True)


I'll help you get the current time! First, let me ask you about your timezone preference.What timezone would you like me to show the current time for? Please provide the timezone in IANA format (e.g., America/New_York, Europe/London, Asia/Tokyo, etc.), or if you're not sure, you can tell me your city or region and I'll help determine the appropriate timezone.

Confirm user cannot continue because LLM is waiting for user input.

In [4]:
resp = await agent.continue_async()
# Stream the response
async for chunk in resp():
    if chunk:
        if isinstance(chunk, str):
            print(chunk, end="", flush=True)

Confirm that "continue_async" with message will work when user responds.

In [5]:
resp = await agent.continue_async(user_message="Use SGT")
# Stream the response
async for chunk in resp():
    if chunk:
        if isinstance(chunk, str):
            print(chunk, end="", flush=True)


Perfect! I'll get the current time in Singapore Time (SGT) for you.The current time in Singapore (SGT) is:

**2025-07-13 15:07:20** (3:07:20 PM)

This is Singapore Standard Time, which is UTC+8.

Confirm if its end of response by continuing the agent.

In [6]:
resp = await agent.continue_async()
# Stream the response
async for chunk in resp():
    if chunk:
        if isinstance(chunk, str):
            print(chunk, end="", flush=True)
